In [74]:
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.stats import norm, t

In [33]:
DATA = Path("..") / "data"
prices = pd.read_csv(DATA / "prices.csv", index_col=0, parse_dates=True)
prices.tail()

,AMZN,GC=F,IWM,JNJ,JPM,NVDA,PG,SPY,VUSTX,XOM
Date,,,,,,,,,,
2026-10-12,259.529999,4190.600098,277.899994,263.730011,334.299988,226.320007,153.199997,778.099976,7.15,172.149994
2026-10-13,258.579987,4177.299805,276.700012,248.610001,331.429993,228.520004,153.039993,778.010010,7.18,174.080002
2026-10-14,254.080002,4107.899902,271.369995,253.910004,335.279999,221.300003,156.529999,769.599976,7.22,177.009995
2026-10-15,254.759995,4170.100098,275.079987,253.279999,335.529999,217.229996,157.080002,771.789978,7.31,174.679993
2026-10-16,249.509995,4191.500000,270.250000,255.520004,337.984985,214.619995,159.070007,762.489990,NaN,178.580002


In [ ]:
"""
Monte Carlo:Jak dokładne jest VaR z symulacji i jak zależy od liczby losowań?	VaR 99% portfela (np. równe wagi 10 aktywów) trzema sposobami: symulacja z rozkładu normalnego, symulacja z historycznych zwrotów (bootstrap) i VaR historyczny. Wykres błędu estymacji VaR w zależności od liczby symulacji (100, 1 000, 10 000, 100 000). Zmienne antytetyczne w jednym prostym przykładzie: porównanie wariancji estymatora ze zmiennymi antytetycznymi i bez nich.
"""

In [37]:
data_returns = np.log(prices / prices.shift(1)).dropna()
data_returns.head()

,AMZN,GC=F,IWM,JNJ,JPM,NVDA,PG,SPY,VUSTX,XOM
Date,,,,,,,,,,
2000-09-05,0.096131,-0.004342,-0.006945,0.006725,0.014326,0.001572,0.018164,-0.008024,-0.000954,0.010551
2000-09-06,0.004096,-0.005818,-0.004365,0.000000,-0.019890,-0.071576,0.014888,-0.011427,-0.004777,-0.004508
2000-09-07,-0.053159,-0.000730,0.014764,0.002009,0.022076,0.052558,-0.012890,0.008530,-0.000958,0.008995
2000-09-08,-0.013015,-0.002558,-0.017392,0.010976,0.009777,-0.061025,0.012890,-0.006860,0.001915,-0.002990
2000-09-11,-0.013187,-0.000732,0.001169,0.005772,-0.005420,-0.041673,-0.011892,-0.001461,-0.003836,0.017621


In [40]:
data_mean = data_returns.mean()
data_mean

AMZN     0.000676
GC=F     0.000391
IWM      0.000296
JNJ      0.000378
JPM      0.000378
NVDA     0.000953
PG       0.000363
SPY      0.000322
VUSTX    0.000161
XOM      0.000359
dtype: float64

In [39]:
data_std = data_returns.std()
data_std

AMZN     0.028776
GC=F     0.011281
IWM      0.015030
JNJ      0.011689
JPM      0.022848
NVDA     0.035062
PG       0.011913
SPY      0.012032
VUSTX    0.007818
XOM      0.016417
dtype: float64

In [75]:
# EQUAL-WEIGHT PORTFOLIO

n = len(prices.columns)
weights = np.asarray([1 / n] * n)

ew_portfolio_returns = data_returns.dot(weights)
ew_portfolio_mean = ew_portfolio_returns.mean()
ew_portfolio_std = ew_portfolio_returns.std()

print(f"Equal-Weight Portfolio mean : {float(ew_portfolio_mean):.5f}")
print(f"Equal-Weight Portfolio std : {float(ew_portfolio_std):.5f}")

Equal-Weight Portfolio mean : 0.00043
Equal-Weight Portfolio std : 0.01054


In [86]:
# Normal Distribution VaR

normal_var = norm.ppf(0.01, loc=ew_portfolio_mean, scale=ew_portfolio_std)
float(normal_var)

-0.02408421479579366

In [87]:
# Monte Carlo VaR

size = 10000
rng = np.random.default_rng(42)
mc_returns = rng.normal(loc=ew_portfolio_mean, scale=ew_portfolio_std, size=size)

monte_carlo_var = np.percentile(boot_returns, 1)
float(monte_carlo_var)

-0.029708076491886143

In [90]:
# Bootstrap VaR

size = 10000
boot_returns = rng.choice(ew_portfolio_returns, size=size, replace=True)
bootstrap_var = np.percentile(boot_returns, 1)
float(bootstrap_var)

-0.029306817781206796

In [89]:
# Historical VaR

historical_var = np.percentile(ew_portfolio_returns, 1)
float(historical_var)

-0.029619421546738463